In [ ]:
# ---------- Environment ----------
class CoinGridEnv(gym.Env):
    def __init__(self, grid_size=5, max_steps=30, layout_mode="fixed", instruction_mode="fixed",varied_instructions=False):
        super().__init__()
        self.grid_size = grid_size
        self.max_steps = max_steps
        self.colors = ['red', 'green', 'blue']
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
        self.layout_mode = layout_mode
        self.instruction_mode = instruction_mode
        self.varied_instructions = varied_instructions


        self.observation_space = spaces.Dict({
            "obs_grid": spaces.Box(low=0, high=3, shape=(2, grid_size, grid_size), dtype=np.float32),
            "instruction_emb": spaces.Box(low=-np.inf, high=np.inf, shape=(384,), dtype=np.float32)
        })

        self.action_space = spaces.Discrete(4)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.agent_pos = [0, 0]
        self.steps = 0
        self.collected = []
        self.grid = np.zeros((self.grid_size, self.grid_size), dtype=np.int32)

        # ---------------- Layouts ----------------
        if self.layout_mode == "fixed":
            # 5 fixed coins, deterministic
            positions = [(1, 1), (2, 2), (3, 3), (4, 4), (1, 3)]
            colors_seq = [1, 2, 3, 1, 2]
            for (x, y), color in zip(positions, colors_seq):
                self.grid[x, y] = color
            color_counts = {'red': 2, 'green': 2, 'blue': 1}

        elif self.layout_mode == "semi-random-3fixed":
            fixed_positions = [(1, 1), (2, 2), (3, 3)]
            fixed_colors = [1, 2, 3]
            color_counts = {color: 0 for color in self.colors}
            for (x, y), color_idx in zip(fixed_positions, fixed_colors):
                self.grid[x, y] = color_idx
                color_counts[self.colors[color_idx - 1]] += 1
            remaining = 5 - len(fixed_positions)
            all_positions = [(x, y) for x in range(self.grid_size) for y in range(self.grid_size)
                             if (x, y) not in fixed_positions and (x, y) != tuple(self.agent_pos)]
            random.shuffle(all_positions)
            for i in range(remaining):
                x, y = all_positions[i]
                color_idx = random.randint(1, len(self.colors))
                self.grid[x, y] = color_idx
                color_counts[self.colors[color_idx - 1]] += 1

        elif self.layout_mode == "semi-random-1fixed":
            fixed_positions = [(1, 1)]
            fixed_colors = [1]
            color_counts = {color: 0 for color in self.colors}
            for (x, y), color_idx in zip(fixed_positions, fixed_colors):
                self.grid[x, y] = color_idx
                color_counts[self.colors[color_idx - 1]] += 1
            remaining = 5 - len(fixed_positions)
            all_positions = [(x, y) for x in range(self.grid_size) for y in range(self.grid_size)
                             if (x, y) not in fixed_positions and (x, y) != tuple(self.agent_pos)]
            random.shuffle(all_positions)
            for i, color in enumerate(self.colors[1:], start=2):
              x, y = all_positions.pop()
              self.grid[x, y] = color_idx = i
              color_counts[color] += 1

    # Fill remaining slots randomly
            for i in range(remaining - 2):
              x, y = all_positions.pop()
              color_idx = random.randint(1, len(self.colors))
              self.grid[x, y] = color_idx
              color_counts[self.colors[color_idx - 1]] += 1


        elif self.layout_mode == "semi-random-local":
            # 3 clustered coins + 2 random
            color_counts = {color: 0 for color in self.colors}
            cx, cy = random.randint(0, self.grid_size-2), random.randint(0, self.grid_size-2)
            local_positions = [(x, y) for x in range(cx, cx+2) for y in range(cy, cy+2) if (x, y) != tuple(self.agent_pos)]
            random.shuffle(local_positions)
            for i in range(3):
                x, y = local_positions[i]
                color_idx = random.randint(1, len(self.colors))
                self.grid[x, y] = color_idx
                color_counts[self.colors[color_idx - 1]] += 1
            all_positions = [(x, y) for x in range(self.grid_size) for y in range(self.grid_size)
                             if (x, y) != tuple(self.agent_pos) and self.grid[x, y] == 0]
            random.shuffle(all_positions)
            for i in range(2):
                x, y = all_positions[i]
                color_idx = random.randint(1, len(self.colors))
                self.grid[x, y] = color_idx
                color_counts[self.colors[color_idx - 1]] += 1

        elif self.layout_mode == "semi-random-scattered":
            # 5 coins placed far apart
            color_counts = {color: 0 for color in self.colors}
            placed = []
            while len(placed) < 5:
                x, y = random.randint(0, self.grid_size - 1), random.randint(0, self.grid_size - 1)
                if (x, y) == tuple(self.agent_pos) or (x, y) in placed:
                    continue
                if any(abs(px - x) + abs(py - y) < 2 for (px, py) in placed):
                    continue
                color_idx = random.randint(1, len(self.colors))
                self.grid[x, y] = color_idx
                color_counts[self.colors[color_idx - 1]] += 1
                placed.append((x, y))

        else:  # fully random
            num_coins = 5
            color_counts = {color: 0 for color in self.colors}
            all_positions = [(x, y) for x in range(self.grid_size) for y in range(self.grid_size)
                             if (x, y) != tuple(self.agent_pos)]
            random.shuffle(all_positions)
            for i, color in enumerate(self.colors):
                x, y = all_positions[i]
                self.grid[x, y] = i + 1
                color_counts[color] += 1
            for i in range(len(self.colors), num_coins):
                x, y = all_positions[i]
                color_idx = random.randint(1, len(self.colors))
                self.grid[x, y] = color_idx
                color_counts[self.colors[color_idx - 1]] += 1

        # ---------------- Instructions ----------------
        
        if self.varied_instructions: 
            available_colors = [c for c in self.colors if color_counts[c] > 0]
            random.shuffle(available_colors)
            num_instruction_colors = random.randint(1, len(available_colors))
            chosen_colors = available_colors[:num_instruction_colors]

            instruction_counts = {}
            for color in chosen_colors:
                max_available = color_counts[color]
                instruction_counts[color] = random.randint(1, max_available)
            templates = [
                "Pick up {parts}",
                "Grab {parts}",
                "Collect exactly {parts}",
                "Your task is to gather {parts}",
                "Ensure you fetch {parts}"
            ]
            template = random.choice(templates)
            parts = []
            for color, count in instruction_counts.items():
                coin_word = "coin" if count == 1 else "coins"
                parts.append(f"{count} {color} {coin_word}")

            # Join parts naturally
            if len(parts) == 1:
                joined_parts = parts[0]
            elif len(parts) == 2:
                joined_parts = f"{parts[0]} and {parts[1]}"
            else:
                joined_parts = f"{', '.join(parts[:-1])}, and {parts[-1]}"

            # Apply template
            self.instruction = template.format(parts=joined_parts)


        elif self.instruction_mode == "fixed":
            self.instruction = "Collect 1 red and 1 green coins"
        elif self.instruction_mode.startswith("templated"):
            if self.instruction_mode == "templated-1color":
                num_colors = 1
            elif self.instruction_mode == "templated-2color":
                num_colors = 2
            elif self.instruction_mode == "templated-3color":
                num_colors = 3
            else:
                num_colors = random.choice([1, 2, 3])
            self.instruction = self._generate_templated_instruction(color_counts, num_colors)
        else:
            available_colors = [c for c in self.colors if color_counts[c] > 0]
            random.shuffle(available_colors)
            num_instruction_colors = random.randint(1, len(available_colors))
            chosen_colors = available_colors[:num_instruction_colors]
            instruction_counts = {}
            for color in chosen_colors:
                max_available = color_counts[color]
                instruction_counts[color] = random.randint(1, max_available)
            parts = [f"{v} {k}" for k, v in instruction_counts.items()]
            self.instruction = "Collect " + " and ".join(parts) + " coin"
            if sum(instruction_counts.values()) > 1:
                self.instruction += "s"

        self.instruction_emb = self.encoder.encode(self.instruction)
        self.instruction_emb = self.instruction_emb / np.linalg.norm(self.instruction_emb)
        return self._get_obs(), {}

    def _generate_templated_instruction(self, color_counts, num_colors):
        available_colors = [c for c in self.colors if color_counts[c] > 0]
        num_colors = min(num_colors, len(available_colors))
        chosen_colors = random.sample(available_colors, k=num_colors)
        instruction_parts = []
        for color in chosen_colors:
            max_available = color_counts[color]
            count = random.randint(1, max_available)
            instruction_parts.append(f"{count} {color}")
        instr = "Collect " + " and ".join(instruction_parts) + " coin"
        if sum(int(p.split()[0]) for p in instruction_parts) > 1:
            instr += "s"
        return instr

    def step(self, action):
        x, y = self.agent_pos
        if action == 0 and x > 0: x -= 1
        elif action == 1 and x < self.grid_size - 1: x += 1
        elif action == 2 and y > 0: y -= 1
        elif action == 3 and y < self.grid_size - 1: y += 1
        self.agent_pos = [x, y]

        color_idx = self.grid[x, y]
        if color_idx > 0:
            color = self.colors[color_idx - 1]
            self.collected.append((color, (x, y)))
            self.grid[x, y] = 0

        self.steps += 1
        done = self.steps >= self.max_steps
        return self._get_obs(), 0.0, done, False, {}

    def _get_obs(self):
        grid_layer = self.grid.copy()
        agent_layer = np.zeros_like(grid_layer)
        agent_layer[self.agent_pos[0], self.agent_pos[1]] = 1
        obs_grid = np.stack([grid_layer, agent_layer]).astype(np.float32)
        return {"obs_grid": obs_grid, "instruction_emb": self.instruction_emb.astype(np.float32)}

    def get_episode_summary(self):
        counts = {}
        for color, _ in self.collected:
            counts[color] = counts.get(color, 0) + 1
        summary = ", ".join(f"{v} {k}" for k, v in counts.items()) or "nothing"
        return f"The agent collected: {summary}."


# ---------- Reward Wrapper ----------
class StepwiseRewardWrapper(gym.Wrapper):
    def __init__(self, env, shaping_alpha=0.08, step_penalty=-0.01):
        super().__init__(env)
        self.required = {}
        self.collected = {}
        self.ep_rwd = []
        self.episode_reward = 0.0
        self.successes = []
        self.shaping_alpha = shaping_alpha
        self.step_penalty = step_penalty
        self.prev_potential = None

    def reset(self, **kwargs):
        self.episode_reward = 0.0
        obs, info = self.env.reset(**kwargs)
        self.instruction = self.env.instruction
        self.required = self._parse_instruction(self.instruction)
        self.collected = {}
        self.prev_potential = self._potential()
        return obs, info


    def _nearest_distance_of_color(self, color):
        color_idx = self.env.colors.index(color) + 1
        ax, ay = self.env.agent_pos
        min_dist = None

        for x in range(self.env.grid_size):
            for y in range(self.env.grid_size):
                if self.env.grid[x, y] == color_idx:
                    d = abs(ax - x) + abs(ay - y)
                    if min_dist is None or d < min_dist:
                        min_dist = d

        return min_dist


    def _potential(self):
        total = 0.0
        for color, req in self.required.items():
            needed = req - self.collected.get(color, 0)

            if needed > 0:
                dist = self._nearest_distance_of_color(color)

                if dist is None:
                    dist = self.env.grid_size * 2  # fallback

                total += needed * dist

        return -self.shaping_alpha * total


    def _compute_final_score(self):
        """Compute final episode score (evaluation only)."""
        missed = 0
        extra = 0

        for color, req in self.required.items():
            collected = self.collected.get(color, 0)

            if collected < req:
                missed += (req - collected)
            elif collected > req:
                extra += (collected - req)

        for color, collected in self.collected.items():
            if color not in self.required:
                extra += collected

        score = 1.0 - 0.5 * missed - 0.25 * extra
        score = max(min(score, 1.0), -1.0)

        return score

    # ------------------- Main step logic ---------------------


    # ---------- Main step logic ----------

    def step(self, action):
        prev_counts = dict(self.collected)
        prev_pot = self.prev_potential   # already set in reset()

        obs, _, done, truncated, info = self.env.step(action)

        # rebuild collected counts
        self.collected = {}
        for (col, _) in self.env.collected:
            self.collected[col] = self.collected.get(col, 0) + 1

    # ---------- Stepwise reward ----------
        reward = 0.0

        for color in self.collected:
            prev = prev_counts.get(color, 0)
            now = self.collected.get(color, 0)

            for _ in range(now - prev):
                current_count = prev + 1

                if color in self.required:
                    if current_count <= self.required[color]:
                        reward += 0.25
                    else:
                        reward -= 0.25
                else:
                    reward -= 0.25

                prev += 1

        # ---------- Potential-based shaping ----------
        new_pot = self._potential()
        reward += (new_pot - prev_pot)
        self.prev_potential = new_pot

        # ---------- Step penalty ----------
        reward += self.step_penalty
        self.episode_reward += reward

        info["stepwise_reward"] = reward

        # ---------- Episode end (evaluation only) ----------
        if done:
            final_score = self._compute_final_score()
            self.ep_rwd.append(final_score)

            # ❗ do NOT override reward
            info["final_reward"] = final_score

            print(f"Final episodic score: {final_score:.3f}")
            print(f"Instruction: {self.instruction}")
            print(f"Summary: {self.env.get_episode_summary()}")
        return obs, reward, done, truncated, info


    def _parse_instruction(self, text):
        matches = re.findall(r'(\d+)\s+(red|green|blue)', text)
        return {color: int(count) for count, color in matches}

class HybridRewardWrapper(gym.Wrapper):
    def __init__(self, env, shaping_alpha=0.08, step_penalty=-0.01, llm_weight=1.0):
        super().__init__(env)
        self.required = {}
        self.collected = {}
        self.ep_rwd = []
        self.shaping_alpha = shaping_alpha
        self.step_penalty = step_penalty
        self.prev_potential = None
        self.llm_weight = llm_weight

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.instruction = self.env.instruction
        self.required = self._parse_instruction(self.instruction)
        self.collected = {}
        self.prev_potential = self._potential()
        return obs, info

 
    def step(self, action):
        prev_counts = dict(self.collected)
        prev_pot = self.prev_potential   # already set in reset()

        obs, _, done, truncated, info = self.env.step(action)

        # rebuild collected counts
        self.collected = {}
        for (col, _) in self.env.collected:
            self.collected[col] = self.collected.get(col, 0) + 1

    # ---------- Stepwise reward ----------
        reward = 0.0

        for color in self.collected:
            prev = prev_counts.get(color, 0)
            now = self.collected.get(color, 0)

            for _ in range(now - prev):
                current_count = prev + 1

                if color in self.required:
                    if current_count <= self.required[color]:
                        reward += 0.25
                    else:
                        reward -= 0.25
                else:
                    reward -= 0.25

                prev += 1

        # -------- Potential shaping --------
        new_pot = self._potential()
        reward += (new_pot - prev_pot)
        self.prev_potential = new_pot

        # -------- Step penalty --------
        reward += self.step_penalty

        # -------- LLM reward (ONLY at end) --------
        if done:
            instruction = self.env.instruction
            summary = self.env.get_episode_summary()

            message = build_prompt(instruction, summary)
            print("Calling LLM (Hybrid)...")

            llm_response = query_ollama(message)
            score = re.search(r"Score:\s*(-?\d+(?:\.\d+)?)", llm_response)
            llm_score = float(score.group(1)) if score else 0.0

            reward += self.llm_weight * llm_score   # ✅ ADD (not replace)

            self.ep_rwd.append(llm_score)
            info["llm_reward"] = llm_score
            print(f"Stage 3.8-Hybrid | {instruction} | {summary} | Score={llm_score:.2f}")
        return obs, reward, done, truncated, info

    def _parse_instruction(self, text):
        matches = re.findall(r'(\d+)\s+(red|green|blue)', text)
        return {color: int(count) for count, color in matches}

    def _nearest_distance_of_color(self, color):
        color_idx = self.env.colors.index(color) + 1
        ax, ay = self.env.agent_pos
        min_dist = None
        for x in range(self.env.grid_size):
            for y in range(self.env.grid_size):
                if self.env.grid[x, y] == color_idx:
                    d = abs(ax - x) + abs(ay - y)
                    if min_dist is None or d < min_dist:
                        min_dist = d
        return min_dist

    def _potential(self):
        total = 0.0
        for color, req in self.required.items():
            needed = req - self.collected.get(color, 0)
            if needed > 0:
                dist = self._nearest_distance_of_color(color)
                if dist is None:
                    dist = self.env.grid_size * 2
                total += needed * dist
        return -self.shaping_alpha * total

def build_prompt(instruction: str, summary: str) -> str: 
    return ( 
        "You are an automated evaluator for a coin-collection RL task.\n\n" 
        
        "TASK:\n" "Evaluate the agent's performance and compute a numeric score.\n\n" 
        
        "SCORING RULES:\n" 
        "- Start with a base score of 1.0.\n" 
        "- For every missed coin, subtract 0.5.\n" 
        "- For every extra or wrong-color coin, subtract 0.25.\n" 
        "- For each color in the instruction:\n" 
            " - If collected < required → missed = required - collected\n" 
            " - If collected > required → extra = collected - required\n" 
        "- For each color not in the instruction but collected → extra = collected\n" 
        "- Final score = 1.0 − 0.5 × total_missed − 0.25 × total_extra\n" 
        "- Clip the final score to the range [−1.0, 1.0].\n\n" 
         
        "PROCESS:\n"
        "1. First, extract all colors and counts from both the instruction and agent’s action.\n"
        "   - Include all colors even if count = 0.\n"
        "2. Compute missed and extra coins *once per color* (do not double count).\n"
        "3. Summarize total_missed and total_extra clearly.\n"
        "4. Then compute and show the final numeric score step by step.\n"
        "5. Do not include extra commentary or text outside the required format.\n\n"
        
        "FORMAT STRICTLY:\n" 
        "Explanation: <your reasoning>\n" 
        #"Don't give any reasoning. Your response should only contain the score.\n"
        "Score: <float between -1.0 and 1.0>\n\n" 
        f"Instruction: {instruction}\n" 
        f"Agent's action: {summary}\n" 

)

class LLMEpisodicRewardWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.ep_rwd = []

    def step(self, action):
        obs, reward, done, truncated, info = self.env.step(action)
        if done:
            instruction = self.env.instruction
            summary = self.env.get_episode_summary()

            message = build_prompt(instruction, summary)
            
            print("Calling LLM for reward...")
            llm_response = query_ollama(message)
            score = re.search(r"Score:\s*(-?\d+(?:\.\d+)?)", llm_response)
            score = float(score.group(1)) if score else None
            reward = score
            info["llm_reward"] = reward
            self.ep_rwd.append(reward)
            print(f"Stage 4-LLM | {instruction} | {summary} | Score={reward:.2f}")

        else:
          reward=0.0

        return obs, reward, done, truncated, info


# ---------- Training by Stage ----------
def train_curriculum_stage(stage: int, model=None, total_timesteps=30000):
    print(f"\n=== Training Stage {stage} ===")

    if stage == "4-llm":
        base_env = CoinGridEnv(layout_mode="semi-random-1fixed", instruction_mode="random", varied_instructions=True)
        env = LLMEpisodicRewardWrapper(base_env)
    elif stage == "3.8-hybrid":
        base_env = CoinGridEnv(layout_mode="semi-random-1fixed", instruction_mode="random", varied_instructions=True)
        env = HybridRewardWrapper(base_env)
    elif stage == "3.5-bridge":
        base_env = CoinGridEnv(layout_mode="semi-random-1fixed", instruction_mode="random", varied_instructions=True)
        env = StepwiseRewardWrapper(base_env, shaping_alpha= 0.08, step_penalty= -0.01)
    else:
      if stage == 1:
        layout_mode, instruction_mode = "fixed", "fixed"
      elif stage == 1.1:
        layout_mode, instruction_mode = "fixed", "templated-1color"
      elif stage == 1.2:
        layout_mode, instruction_mode = "fixed", "templated-2color"
      elif stage == 1.3:
        layout_mode, instruction_mode = "fixed", "templated-3color"
      elif stage == 2:
        layout_mode, instruction_mode = "fixed", "random"
      elif stage == 2.1:
        layout_mode, instruction_mode = "semi-random-3fixed", "fixed"
      elif stage == 2.2:
        layout_mode, instruction_mode = "semi-random-1fixed", "fixed"
      elif stage == 3.1:
        layout_mode, instruction_mode = "semi-random-local", "random"
      elif stage == 3.2:
        layout_mode, instruction_mode = "semi-random-scattered", "random"
      elif stage == 3:
        layout_mode, instruction_mode = "random", "fixed"
      else:  # stage 4
        layout_mode, instruction_mode = "random", "random"

      base_env = CoinGridEnv(layout_mode=layout_mode, instruction_mode=instruction_mode)
      env = StepwiseRewardWrapper(base_env, shaping_alpha= 0.08, step_penalty= -0.01)
    vec_env = make_vec_env(lambda: env, n_envs=1)
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.)


    if model is None:
        model = PPO(
            "MultiInputPolicy",
             vec_env,
             verbose=1,
             gamma=0.99,          # discount
             n_steps=2048,        # rollout size per env
             batch_size=64,       # minibatch size
             gae_lambda=0.95,
             ent_coef=0.01,       # encourage exploration
             learning_rate=3e-4,
             n_epochs=10,
        )
    else:
        model.set_env(vec_env)

    model.learn(total_timesteps=total_timesteps)
    return model, env.ep_rwd


# ---------- Run Curriculum ----------
stage_timesteps = {
    1: 10000, 1.1: 15000, 1.2: 20000, 1.3: 30000,
    2: 50000, 2.1: 60000, 2.2: 75000, "3.5-bridge": 90000, 
    "3.8-hybrid": 50000, "4-llm": 10000,
}

stagewise_rewards = {}
model = None
for stage in [1, 1.1, 1.2, 1.3, 2, 2.1, 2.2, "3.5-bridge", "3.8-hybrid", "4-llm"]:
    timesteps = stage_timesteps[stage]
    model, stagewise_rewards[stage] = train_curriculum_stage(stage, model=model, total_timesteps=timesteps)

# ---------- Plotting ----------
window = 50

plt.figure(figsize=(12, 5))

stage_boundaries = {}
cumulative_episodes = 0

selected_stages = ["3.5-bridge", "3.8-hybrid", "4-llm"]  # ← choose your stages

for stage, rewards in stagewise_rewards.items():

    if stage not in selected_stages:
        continue  # ← skip unwanted stages

    episodes = list(range(1, len(rewards) + 1))

    # Apply smoothing
    smoothed = rewards
    if len(rewards) >= window:
        smoothed = np.convolve(rewards, np.ones(window) / window, mode="valid")
        episodes = list(range(window, len(rewards) + 1))

    offset_eps = [e + cumulative_episodes for e in episodes]

    # Highlight (same logic)
    if stage in ["3.5-bridge", "3.8-hybrid", "4-llm"]:
        plt.plot(offset_eps, smoothed, label=f"Stage {stage}", linewidth=2.5)
    else:
        plt.plot(offset_eps, smoothed, label=f"Stage {stage}", alpha=0.6)

    cumulative_episodes += len(rewards)
    stage_boundaries[stage] = cumulative_episodes

# Vertical lines
for boundary in stage_boundaries.values():
    plt.axvline(boundary, linestyle="--", alpha=0.5)

plt.title("Training Performance Across Selected Curriculum Stages")
plt.xlabel("Episodes")
plt.ylabel("Normalized Reward")

plt.ylim(-0.2, 1.0)
plt.legend(fontsize=8, ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def build_prompt(instruction: str, summary: str) -> str: 
    return ( 
        "You are an automated evaluator for a coin-collection RL task.\n\n" 
        "TASK:\n" "Evaluate the agent's performance and compute a numeric score.\n\n" 
        
        "SCORING RULES:\n" 
        "- Start with a base score of 1.0.\n" 
        "- Apply penalties for performance errors as follows:\n" 
        "  - Deduct points for each missed coin (when the agent collects fewer coins of a color than required).\n" 
        "  - Deduct points for each extra or wrong-color coin (when the agent collects more coins than required or collects colors not mentioned in the instruction).\n" 
        "- The penalties should reflect the relative severity of the mistakes, where missing required coins is worse than collecting extra ones.\n"
        "- The final score should be clipped to the range [−1.0, 1.0].\n\n" 
         
        "PROCESS:\n"
        "1. First, extract all colors and counts from both the instruction and agent’s action.\n"
        "   - Include all colors even if count = 0.\n"
        "2. Compute missed and extra coins once per color (do not double count).\n"
        "3. Summarize total_missed and total_extra clearly.\n"
        "4. Then compute and show the final numeric score step by step based on your chosen penalty logic.\n"
        "5. Do not include extra commentary or text outside the required format. But please include the calculations for the score\n\n"
        
        
        "FORMAT STRICTLY:\n" 
        "Explanation: <your reasoning>\n" 
        #"Don't give any reasoning. Your response should only contain the score.\n"
        "FScore: <float between -1.0 and 1.0>\n\n" 
        f"Instruction: {instruction}\n" 
        f"Agent's action: {summary}\n" 

)

In [ ]:
def build_prompt(instruction: str, summary: str) -> str: 
    return ( 
        "You are an automated evaluator for a coin-collection RL task.\n\n" 
        "TASK:\n" "Evaluate the agent's performance and compute a numeric score.\n\n" 
        
        "SCORING RULES:\n" 
        "- Start with a base score of 1.0.\n" 
        "- Apply penalties for performance errors as follows:\n" 
        "  - Deduct points for each missed coin (when the agent collects fewer coins of a color than required).\n" 
        "  - Deduct points for each extra or wrong-color coin (when the agent collects more coins than required or collects colors not mentioned in the instruction).\n" 
        "  - The penalties should reflect the relative severity of the mistakes, ensuring that deductions are based on the overall accuracy and completeness of the agent’s collection..\n"
        "  - The final score should be clipped to the range [−1.0, 1.0].\n\n" 
         
        "PROCESS:\n"
        "1. First, extract all colors and counts from both the instruction and agent’s action.\n"
        "   - Include all colors even if count = 0.\n"
        "2. Compute missed and extra coins once per color (do not double count).\n"
        "3. Summarize total_missed and total_extra clearly.\n"
        "4. Then compute and show the final numeric score step by step based on your chosen penalty logic.\n"
        "5. Do not include extra commentary or text outside the required format. But please include the calculations for the score\n\n"
        
        
        "FORMAT STRICTLY:\n" 
        "Explanation: <your reasoning>\n" 
        #"Don't give any reasoning. Your response should only contain the score.\n"
        "FScore: <float between -1.0 and 1.0>\n\n" 
        f"Instruction: {instruction}\n" 
        f"Agent's action: {summary}\n" 

)

In [ ]:
class LLMRewardWrapper(gym.Wrapper):
    """
    One unified wrapper:
    - At each step: sends instruction + step event to LLM → returns numeric reward  
    - At episode end: sends final summary → returns episodic reward  
    - No hard-coded values
    """

    def __init__(self, env):
        super().__init__(env)
        self.prev_counts = {}
        self.collected = {}
        self.episode_step_events = []   # Store step descriptions for final prompt
        self.ep_rwd = []

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.prev_counts = {}
        self.collected = {}
        self.episode_step_events = []
        self.instruction = self.env.instruction
        return obs, info

    # -------------------------------------------------------
    # Helper: Build a prompt for step reward
    # -------------------------------------------------------
    def build_step_prompt(self, instruction, prev_counts, new_counts, just_collected):
        return f"""
You are evaluating an RL agent step-by-step in a coin collection grid world.

Instruction:
{instruction}

Previous collected counts:
{prev_counts}

New collected counts after this step:
{new_counts}

Coins collected *on this exact step*:
{just_collected}

TASK:
Give a numeric reward between -1.0 and 1.0 reflecting whether this single action
moved the agent toward completing the instruction.

GUIDELINES:
- Positive reward if the collected coin is useful.
- Negative reward if extra/wrong coins are collected.
- If no coin collected, give small positive/negative reward ONLY if helpful/harmful.
- Do NOT give reasoning.

FORMAT:
FScore: <numeric value>
"""

    # -------------------------------------------------------
    # Helper: Build episodic reward prompt
    # -------------------------------------------------------
    def build_episode_prompt(self, instruction, summary):
        return f"""
You are evaluating the agent's entire episode in a coin collection task.

Instruction:
{instruction}

Final summary of collected coins:
{summary}

Step-by-step events:
{self.episode_step_events}

TASK: m
Give a final numeric reward ∈ [-1.0, 1.0] based on how well the agent satisfied the instruction.
No reasoning.

FORMAT:
FScore: <numeric value>
"""

    # -------------------------------------------------------
    # Main Step Function
    # -------------------------------------------------------
    def step(self, action):
        prev = dict(self.collected)

        obs, _, done, truncated, info = self.env.step(action)

        # Recompute collected dict from environment state
        self.collected = {}
        for (color, _) in self.env.collected:
            self.collected[color] = self.collected.get(color, 0) + 1

        # Determine what was collected on this step
        just_collected = []
        for color, new_ct in self.collected.items():
            prev_ct = prev.get(color, 0)
            if new_ct > prev_ct:
                for _ in range(new_ct - prev_ct):
                    just_collected.append(color)

        # ----- LLM STEPWISE REWARD -----
        step_prompt = self.build_step_prompt(
            self.instruction,
            prev,
            self.collected,
            just_collected
        )

        llm_response = query_ollama(step_prompt)
        score = re.findall(r"FScore:\s*(-?\d+(?:\.\d+)?)", llm_response)
        step_reward = float(score[-1]) if score else 0.0

        # Log step event for final episode summary
        self.episode_step_events.append(
            f"Step: collected={just_collected}, reward={step_reward}"
        )

        info["llm_step_reward"] = step_reward

        # If done → let LLM compute final episodic reward
        if done:
            final_prompt = self.build_episode_prompt(
                self.instruction,
                self.env.get_episode_summary()
            )
            final_response = query_ollama(final_prompt)
            score = re.findall(r"FScore:\s*(-?\d+(?:\.\d+)?)", final_response)
            final_reward = float(score[-1]) if score else 0.0

            info["llm_final_reward"] = final_reward
            self.ep_rwd.append(final_reward)
            print("\n=== EPISODE END ===")
            print(final_prompt)
            print("LLM Final Reward:", final_reward)

            return obs, final_reward, done, truncated, info

        # Not done → return step reward only
        return obs, step_reward, done, truncated, info

In [ ]:
class LLMRewardWrapper(gym.Wrapper):
    """
    LLM-based reward wrapper with:
    - Correct coin reward signals
    - Negative reward for wrong coins
    - Distance-based shaping reward
    - Stable strict prompting
    """

    def __init__(self, env):
        super().__init__(env)
        self.collected = {}
        self.prev_counts = {}
        self.episode_step_events = []
        self.last_pos = None
        self.ep_rwd = []

    # -------------------------------------------------------
    # HELPER: Manhattan distance
    # -------------------------------------------------------
    def manhattan(self, a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    # -------------------------------------------------------
    # BUILD STEPWISE PROMPT
    # -------------------------------------------------------
    def build_step_prompt(
        self,
        instruction,
        prev_counts,
        new_counts,
        just_collected,
        prev_pos,
        new_pos,
        required,
        coin_positions,
        prev_dist,
        new_dist
    ):
        return f"""
Evaluate a single action in a grid-world coin collection task.

=== TASK ===
{instruction}

=== REQUIRED COINS ===
{required}

=== COIN POSITIONS ===
{coin_positions}

=== AGENT MOVE ===
Previous position: {prev_pos}
New position: {new_pos}

Distance to nearest REQUIRED coin before move: {prev_dist}
Distance after move: {new_dist}

=== COUNTS ===
Previously collected: {prev_counts}
New collected: {new_counts}
Coins collected this step: {just_collected}


=== INSTRUCTIONS FOR REWARD ===
1. If a required coin is collected, give a positive reward.
2. If a wrong/extra coin is collected, give a negative reward.
3. If no coin collected:
   - If agent moved closer to nearest required coin (if new_dist is less than prev_dist)→ small positive reward
   - If agent moved farther from nearest required coin (if new_dist is greater than prev_dist)→ small negative reward
   - If distance did not change → neutral reward
4. Output only the numeric reward as a float.

=== FORMAT ===
FScore: <float>
"""

    # -------------------------------------------------------
    # EPISODIC PROMPT
    # -------------------------------------------------------
    def build_episode_prompt(self, instruction, summary, step_events):
        return f"""
Evaluate the final episode.

Instruction:
{instruction}

Final summary:
{summary}

Step events:
{step_events}

You are an automated evaluator for a coin-collection RL task.

TASK:
Evaluate the agent's performance and compute a numeric score.

SCORING RULES:
- Start with a base score of 1.0.
- For every missed required coin, subtract 0.5.
- For every extra or wrong-color coin, subtract 0.25.
- For each color in the instruction:
    - If collected < required → missed = required - collected
    - If collected > required → extra = collected - required
- For each color not in the instruction but collected → extra = collected
- Final score = 1.0 − 0.5 × total_missed − 0.25 × total_extra
- Clip the final score to the range [−1.0, 1.0].

PROCESS:
1. Extract all colors and counts from the instruction and summary.
2. Compute missed and extra coins once per color.
3. Summarize total_missed and total_extra.
4. Compute final score strictly using the formula.
5. Do not include unnecessary commentary.

FORMAT STRICTLY:
FScore: <float between -1.0 and 1.0>
"""

       

    # -------------------------------------------------------
    # RESET
    # -------------------------------------------------------
    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)

        self.collected = {}
        self.prev_counts = {}
        self.episode_step_events = []

        self.instruction = self.env.instruction
        self.required = self.env.required_coins       # {"blue": 1}
        self.coin_positions = self.env.coin_positions # {"red":[...], ...}

        self.last_pos = tuple(self.env.agent_pos)

        return obs, info

    # -------------------------------------------------------
    # STEP
    # -------------------------------------------------------
    def step(self, action):

        prev_pos = tuple(self.env.agent_pos)

        # Run environment
        prev = dict(self.collected)
        obs, _, done, truncated, info = self.env.step(action)

        # Update counts
        self.collected = {}
        for (color, _) in self.env.collected:
            self.collected[color] = self.collected.get(color, 0) + 1

        # Coins collected in this step
        just_collected = []
        for color, new_ct in self.collected.items():
            prev_ct = prev.get(color, 0)
            if new_ct > prev_ct:
                just_collected.extend([color] * (new_ct - prev_ct))

        new_pos = tuple(self.env.agent_pos)

        # ---------------------------------------------------
        # Distance to nearest required coin
        # ---------------------------------------------------
        required_locations = []
        for col, needed in self.required.items():
            if needed > 0 and col in self.coin_positions:
                required_locations.extend(self.coin_positions[col])

        if required_locations:
            prev_dist = min(self.manhattan(prev_pos, p) for p in required_locations)
            new_dist = min(self.manhattan(new_pos, p) for p in required_locations)
        else:
            prev_dist = new_dist = 0  # no required coins exist

        # ---------------------------------------------------
        # BUILD PROMPT FOR LLM
        # ---------------------------------------------------
        step_prompt = self.build_step_prompt(
            self.instruction,
            prev,
            self.collected,
            just_collected,
            prev_pos,
            new_pos,
            self.required,
            self.coin_positions,
            prev_dist,
            new_dist
        )

        llm_response = query_ollama(step_prompt)
        score = re.findall(r"FScore:\s*(-?\d+(?:\.\d+)?)", llm_response)
        step_reward = float(score[-1]) if score else 0.0
        step_reward = max(-1.0, min(1.0, step_reward))

        print(f"Stepwise Reward: collected={just_collected}, reward={step_reward}")

        # Log event
        self.episode_step_events.append(
            f"Step: {prev_pos}->{new_pos}, collected={just_collected}, reward={step_reward}"
        )
        #print(f"Stepwise Reward : collected={just_collected}, reward={step_reward}")
        # ---------------------------------------------------
        # EPISODE END
        # ---------------------------------------------------
        if done:
            summary = self.env.get_episode_summary()

            final_prompt = self.build_episode_prompt(
                self.instruction,
                summary,
                self.episode_step_events
            )

            final_resp = query_ollama(final_prompt)
            final_score = re.findall(r"FScore:\s*(-?\d+(?:\.\d+)?)", final_resp)
            final_reward = float(final_score[-1]) if final_score else 0.0
            final_reward = max(-1.0, min(1.0, final_reward))

            print("\n===== EPISODE END =====")
            print("Instruction:", self.instruction)
            print("Summary:", summary)
            print("Final Reward:", final_reward)
            print("========================")

            info["llm_final_reward"] = final_reward
            self.ep_rwd.append(final_reward)
            return obs, final_reward, done, truncated, info

        # ---------------------------------------------------
        # NORMAL STEP
        # ---------------------------------------------------
        info["llm_step_reward"] = step_reward
        return obs, step_reward, done, truncated, info
        

In [ ]:
class LLMRewardWrapper(gym.Wrapper):
    """
    LLM-based reward wrapper.
    Python computes ALL numeric facts.
    LLM decides reward based on symbolic state.
    """

    def __init__(self, env):
        super().__init__(env)
        self.collected = {}               # all coins ever collected
        self.required_collected = {}      # ONLY coins that satisfy task
        self.episode_step_events = []
        self.ep_rwd = []

    # -------------------------------------------------------
    # Manhattan distance
    # -------------------------------------------------------
    def manhattan(self, a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    # -------------------------------------------------------
    # Remaining required coins (FACT)
    # -------------------------------------------------------
    def remaining_required(self):
        rem = {}
        for col, needed in self.required.items():
            collected = self.required_collected.get(col, 0)
            if needed - collected > 0:
                rem[col] = needed - collected
        return rem

    # -------------------------------------------------------
    # Remaining coin positions (FACT)
    # -------------------------------------------------------
    def remaining_coin_positions(self):
        remaining = {}
        collected_positions = set(pos for _, pos in self.env.collected)

        for color, positions in self.coin_positions.items():
            rem_pos = [p for p in positions if p not in collected_positions]
            if rem_pos:
                remaining[color] = rem_pos

        return remaining

    # -------------------------------------------------------
    # STEP PROMPT
    # -------------------------------------------------------
    def build_step_prompt(
        self,
        instruction,
        prev_pos,
        new_pos,
        collected_info,
        remaining_required,
        remaining_positions,
        distance_trend
    ):
        return f"""
You are evaluating ONE action in a grid-world coin collection task.

TASK INSTRUCTION:
{instruction}

REMAINING REQUIRED COINS:
{remaining_required}

REMAINING COIN POSITIONS:
{remaining_positions}

AGENT MOVE:
Previous position: {prev_pos}
New position: {new_pos}

COLLECTED THIS STEP:
{collected_info}

DISTANCE TREND (system-computed):
{distance_trend}

STRICT SCORING RULES:
1. If ANY collected coin has type == "extra" → reward = -0.35
2. ELSE IF ANY collected coin has type == "required" → reward = +0.35
3. ELSE (no coin collected):
   - distance_trend == closer → +0.1
   - distance_trend == farther → -0.1
   - distance_trend == same → -0.025
   - distance_trend == no_required → 0.0

OUTPUT FORMAT (STRICT):
FScore: <float>
"""

    # -------------------------------------------------------
    # EPISODE PROMPT
    # -------------------------------------------------------
    def build_episode_prompt(self, instruction, summary, step_events):
        return f"""
Evaluate the final episode.

Instruction:
{instruction}

Final summary:
{summary}

Step events:
{step_events}

SCORING RULES:
- Start with score = 1.0
- For each missed required coin: -0.5
- For each extra coin: -0.25
- Clip to [-1.0, 1.0]

FORMAT STRICTLY:
FScore: <float>
"""

    # -------------------------------------------------------
    # RESET
    # -------------------------------------------------------
    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)

        self.collected = {}
        self.required_collected = {}
        self.episode_step_events = []

        self.instruction = self.env.instruction
        self.required = self.env.required_coins
        self.coin_positions = self.env.coin_positions

        return obs, info

    # -------------------------------------------------------
    # STEP
    # -------------------------------------------------------
    def step(self, action):

        prev_pos = tuple(self.env.agent_pos)

        # ===== FACTS BEFORE STEP =====
        remaining_before = self.remaining_required()
        remaining_positions_before = self.remaining_coin_positions()
        required_locations = []
        for c, remaining_ct in remaining_before.items():
            if remaining_ct > 0:
                required_locations.extend(self.coin_positions.get(c, []))

        if not required_locations:
            prev_dist = None
        else:
            prev_dist = min(
                self.manhattan(prev_pos, p) for p in required_locations
            )

        prev_collected = dict(self.collected)

        # ===== ENV STEP =====
        obs, _, done, truncated, info = self.env.step(action)
        new_pos = tuple(self.env.agent_pos)

        # ===== UPDATE COLLECTED (RAW) =====
        self.collected = {}
        for color, _ in self.env.collected:
            self.collected[color] = self.collected.get(color, 0) + 1

        # Coins collected this step
        just_collected = []
        for c, new_ct in self.collected.items():
            prev_ct = prev_collected.get(c, 0)
            if new_ct > prev_ct:
                just_collected.extend([c] * (new_ct - prev_ct))

        # ===== UPDATE REQUIRED_COLLECTED (TASK-SAFE) =====
        for c in just_collected:
            if self.required.get(c, 0) > self.required_collected.get(c, 0):
                self.required_collected[c] = self.required_collected.get(c, 0) + 1

        # ===== DISTANCE AFTER STEP (same targets) =====
        if not required_locations:
            new_dist = None
        else:
            new_dist = min(
                self.manhattan(new_pos, p) for p in required_locations
            )

        # ===== DISTANCE TREND (PYTHON, NOT LLM) =====
        if prev_dist is None:
            distance_trend = "no_required"
        elif new_dist < prev_dist:
            distance_trend = "closer"
        elif new_dist > prev_dist:
            distance_trend = "farther"
        else:
            distance_trend = "same"

        # ===== CLASSIFY COLLECTED =====
        collected_info = []
        for c in just_collected:
            if c in self.required and self.required_collected.get(c, 0) <= self.required[c] - 1:
                collected_info.append({"color": c, "type": "required"})
            else:
                collected_info.append({"color": c, "type": "extra"})

        # ===== BUILD PROMPT =====
        step_prompt = self.build_step_prompt(
            self.instruction,
            prev_pos,
            new_pos,
            collected_info,
            remaining_before,
            remaining_positions_before,
            distance_trend
        )

        llm_response = query_ollama(step_prompt)
        score = re.findall(r"FScore:\s*(-?\d+(?:\.\d+)?)", llm_response)
        step_reward = float(score[-1]) if score else 0.0
        step_reward = max(-1.0, min(1.0, step_reward))

        print(f"[LLM STEP] {prev_pos}->{new_pos} | collected={collected_info} | trend={distance_trend} | reward={step_reward}")

        self.episode_step_events.append(
            f"{prev_pos}->{new_pos}, collected={collected_info}, trend={distance_trend}, reward={step_reward}"
        )

        # ===== EPISODE END =====
        if done:
            summary = self.env.get_episode_summary()
            final_prompt = self.build_episode_prompt(
                self.instruction, summary, self.episode_step_events
            )

            final_resp = query_ollama(final_prompt)
            final_score = re.findall(r"FScore:\s*(-?\d+(?:\.\d+)?)", final_resp)
            final_reward = float(final_score[-1]) if final_score else 0.0
            final_reward = max(-1.0, min(1.0, final_reward))

            info["llm_final_reward"] = final_reward
            self.ep_rwd.append(final_reward)

            print("\n===== EPISODE END =====")
            print(final_prompt)
            print("Final Reward:", final_reward)
            print("========================")

            return obs, final_reward, done, truncated, info

        info["llm_step_reward"] = step_reward
        return obs, step_reward, done, truncated, info